# Hamiltonians

Hamiltonians are the central objects in quantum mechanics — they describe the energy of a system and govern its time evolution via the Schrödinger equation. In the context of quantum computing, Hamiltonians are used to describe the interactions between qubits and to construct unitary gates via exponentiation.

In general, we can write a Hamiltonian as a linear combination of $M$ local operators:

$$H = \sum_{m=1}^{M} c_m O_m$$

where the coefficients $c_m$ are real and $O_m$ are Hermitian operators (typically Pauli strings for qubit systems).

While there are other packages that provide sophisticated tools for constructing and manipulating Hamiltonians (e.g. symbolic algebra, fermion-to-qubit mappings), Quax keeps things simple: **Hamiltonians are nothing more than `Operator` objects**, and you build them using standard arithmetic on gates.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import jax.numpy as jnp

import quax as qx
from quax.gates import I, X, Y, Z

## Constructing Hamiltonians from Pauli strings

The simplest Hamiltonians are built from Pauli operators. The `|` operator in Quax denotes the tensor product, so `X | X` means $X \otimes X$ — a 2-qubit operator acting as Pauli-X on both qubits simultaneously.

Let's construct the exchange-type Hamiltonian $H = XX + YY$, which appears in models of spin-spin coupling:

In [ ]:
hamiltonian = (X | X) + (Y | Y)
print(hamiltonian)

The resulting object is a `Unitary`, since both $XX$ and $YY$ happen to be unitary (they square to the identity). In general though, a weighted sum of unitaries will produce an `Operator`.

**Operator precedence note:** In Python, the `|` operator binds *less tightly* than `+` and `-`. This means `X | X + Y | Y` would be parsed as `X | (X + Y) | Y`, which is not what we want. Always use parentheses: `(X | X) + (Y | Y)`.

Let's add a coupling strength $\theta$:

In [ ]:
theta = jnp.pi / 4
hamiltonian = theta * ((X | X) + (Y | Y))
print(hamiltonian)

## Extending to multi-qubit systems

Unlike some packages, in Quax, operators are **not** attached to qubit indices — there is no concept of a "qubit register" to which operators are assigned. Instead, you construct the full Pauli string explicitly by tensoring in identity operators on the qubits that are not involved.

For example, to place our $XX + YY$ interaction on qubits 1 and 2 of a 5-qubit system, we tensor identity operators on qubits 0, 3, and 4:

In [ ]:
hamiltonian = theta * ((I | X | X | I | I) + (I | Y | Y | I | I))
print(hamiltonian)

## Exponentiating Hamiltonians

The workhorse of quantum dynamics is Hamiltonian exponentiation. Time evolution under a Hamiltonian $H$ for time $t$ is given by the unitary:

$$U = e^{-iHt}$$

Quax provides two functions for this:
- **`qx.exp(A)`** — computes $e^A$ for any operator $A$. The result is a `Unitary` when the input is anti-Hermitian, and a general `Operator` otherwise.
- **`qx.cis(H)`** — computes $e^{iH}$, analogous to the mathematical function $\text{cis}(x) = e^{ix}$. Since $H$ is typically Hermitian, the output is guaranteed to be unitary.

Using `cis` is often preferable because it makes the unitarity of the result explicit.

### Constructing single-qubit rotation gates

The standard single-qubit rotations are defined as:

$$R_X(\theta) = e^{-i \frac{\theta}{2} X}, \qquad R_Y(\theta) = e^{-i \frac{\theta}{2} Y}, \qquad R_Z(\theta) = e^{-i \frac{\theta}{2} Z}$$

Let's construct these for $\theta = \pi/2$ using `cis`:

In [ ]:
theta = jnp.pi / 2
SX = qx.cis((-theta / 2) * X)
SY = qx.cis((-theta / 2) * Y)
SZ = qx.cis((-theta / 2) * Z)

for name, gate in [("RX(𝜋/2)", SX), ("RY(𝜋/2)", SY), ("RZ(𝜋/2)", SZ)]:
    print(f"{name} =")
    print(jnp.round(gate.matrix, 6))
    print()

### Constructing two-qubit entangling gates

Many important two-qubit gates can be expressed as exponentials of Pauli Hamiltonians. Here are some common examples using familiar conventions:

$$\mathrm{CZ} = e^{-i \frac{\pi}{4}(ZZ - ZI - IZ)}$$

$$\mathrm{CNOT} = e^{-i \frac{\pi}{4}(ZX - ZI - IX)}$$

$$\mathrm{ISWAP} = e^{+i \frac{\pi}{4}(XX + YY)}$$

$$\mathrm{FSIM}(\theta, \phi) = e^{i\frac{\theta}{2}(XX + YY) - i\frac{\phi}{4}(ZZ - ZI - IZ)}$$

The fSIM gate is particularly important as it is the native entangling gate on many superconducting quantum processors, continuously parameterized by the iSWAP angle $\theta$ and the CZ phase $\phi$.

*Note: We use Rigetti sign conventions here, which may differ from other references.*

In [ ]:
hamiltonian = (+jnp.pi / 4) * ((Z | Z) - (I | Z) - (Z | I))
cz = qx.cis(hamiltonian)
print("CZ = ")
print(f"{jnp.round(cz.matrix, 6)}")
print()

hamiltonian = (+jnp.pi / 4) * ((Z | X) - (I | X) - (Z | I))
cx = qx.cis(hamiltonian)
print("CX = ")
print(f"{jnp.round(cx.matrix, 6)}")
print()

hamiltonian = (+jnp.pi / 4) * ((X | X) + (Y | Y))
iswap = qx.cis(hamiltonian)
print("ISWAP = ")
print(f"{jnp.round(iswap.matrix, 6)}")
print()

hamiltonian = (+jnp.pi / 4) * ((X | X) + (Y | Y)) + (+jnp.pi / 4 / 6) * ((Z | Z) - (I | Z) - (Z | I))
fsim = qx.cis(hamiltonian)
print("FSIM(𝜃=𝜋, 𝜙=𝜋/6) = ")
print(f"{jnp.round(fsim.matrix, 6)}")
print()

## Ensembles of Hamiltonians

Like all objects in Quax, Hamiltonian exponentiation supports **batch operations**. By passing an array of coupling strengths $\theta$, we can generate an entire ensemble of unitaries in a single call — this is much more efficient than looping in Python, as JAX can vectorize the computation on GPU/TPU.

Here we sweep $\theta$ from $0$ to $\pi/4$ and generate 24 unitaries of the $XX + YY$ interaction:

In [ ]:
theta = jnp.linspace(0, jnp.pi / 4, 24)
unitaries = qx.cis(theta * ((X | X) + (Y | Y)))
print(unitaries)

## Qudit Hamiltonians

Everything above generalizes naturally to **qutrits** (3-level systems) and higher-dimensional qudits. Quax provides generalized Pauli operators for qutrits — `TX` (cyclic shift), `TY`, and `TZ` (clock matrix) — as well as the eight **Gell-Mann matrices** (`GELLMANN1`–`GELLMANN8`), which form the generators of $\mathfrak{su}(3)$ and play the role that Pauli matrices play for qubits.

Subspace rotation gates like `TRX01(φ)`, `TRY12(φ)`, etc. allow rotations within specific pairs of levels, which is essential for controlling leakage in transmon-based systems where the third level is a spectator.

In [ ]:
from quax.gates import GELLMANN1, GELLMANN3, GELLMANN8, TX, TY, W00

# --- Qutrit Pauli-like Hamiltonians ---

# Two-qutrit coupling using generalized Pauli operators (analogous to XX + YY for qubits).
# W00 is the 3×3 identity (Weyl operator W_{0,0}).
I3 = W00
theta_qt = jnp.pi / 6
H_qutrit = theta_qt * ((TX | TX) + (TY | TY))

U_qutrit = qx.cis(H_qutrit)
print("Two-qutrit unitary from TX-TX + TY-TY coupling:")
print(U_qutrit)
print(f"  Is unitary: {qx.is_unitary(U_qutrit)}")

### Gell-Mann Hamiltonians

The Gell-Mann matrices $\lambda_1, \dots, \lambda_8$ are the standard generators of $\mathfrak{su}(3)$. Any traceless Hermitian $3 \times 3$ operator — and therefore any qutrit Hamiltonian (up to an energy offset) — can be written as a linear combination of these matrices. This is completely analogous to expanding qubit Hamiltonians in the Pauli basis.

In [ ]:
# A single-qutrit Hamiltonian built from Gell-Mann matrices.
# GELLMANN3 ∝ diag(1, -1, 0) and GELLMANN8 ∝ diag(1, 1, -2)/√3
# capture the two independent diagonal generators of su(3).
H_gellmann = 0.5 * GELLMANN3 + 0.3 * GELLMANN8

print("Single-qutrit Gell-Mann Hamiltonian:")
print(f"  {jnp.round(H_gellmann.matrix, 4)}")

U_gm = qx.cis(H_gellmann)
print("\nexp(-i H):")
print(f"  {jnp.round(U_gm.matrix, 4)}")
print(f"  Is unitary: {qx.is_unitary(U_gm)}")

### Subspace rotations

Transmon qubits are really qutrits (or higher) — the third level $|2\rangle$ is always present and can participate in leakage errors. Quax provides subspace rotation gates `TRX01`, `TRY12`, etc. that rotate within a specific pair of energy levels while leaving the third level unchanged. These can be combined into Hamiltonians that model subspace-selective driving.

In [ ]:
# Subspace rotation: π/2 X-rotation in the |0⟩–|1⟩ subspace (leaves |2⟩ alone)
U_01 = qx.gates.TRX01(jnp.pi / 2)
print("TRX01(π/2) — rotation in |0⟩–|1⟩ subspace:")
print(f"  {jnp.round(U_01.matrix, 4)}")

# Compare with a rotation in the |1⟩–|2⟩ subspace
U_12 = qx.gates.TRX12(jnp.pi / 2)
print("\nTRX12(π/2) — rotation in |1⟩–|2⟩ subspace:")
print(f"  {jnp.round(U_12.matrix, 4)}")

# Compose the two for a qutrit circuit: drive 0↔1 then 1↔2
U_composed = U_12 @ U_01
print("\nComposed gate (TRX12 ∘ TRX01):")
print(f"  {jnp.round(U_composed.matrix, 4)}")
print(f"  Is unitary: {qx.is_unitary(U_composed)}")

### Mixed qubit–qutrit systems

Tensor products mix dimensions freely. Here we build a Hamiltonian coupling a qubit (dim 2) to a qutrit (dim 3), acting on a $2 \times 3 = 6$-dimensional Hilbert space.

In [ ]:
# Qubit–qutrit coupling: X ⊗ GELLMANN1 (a 6×6 operator)
H_mixed = 0.4 * (X | GELLMANN1)
print(f"Qubit–qutrit Hamiltonian: {H_mixed}")
print(f"  dims: {H_mixed.dims}")
print(f"  matrix shape: {H_mixed.matrix.shape}")

U_mixed = qx.cis(H_mixed)
print(f"\nResulting unitary: {U_mixed}")
print(f"  Is unitary: {qx.is_unitary(U_mixed)}")